In [1]:
import pandas as pd
import numpy as np
import lightgbm as lgb
from sklearn.model_selection import train_test_split

df = pd.read_parquet("../data/cars_clean.parquet")

df["age"] = (2023 - df["year"]).clip(lower=0)
df["miles_per_year"] = df["mileage"] / df["age"].clip(lower=1)

In [2]:
df.columns

Index(['manufacturer', 'model', 'year', 'mileage', 'drivetrain', 'fuel_type',
       'accidents_or_damage', 'one_owner', 'personal_use_only', 'seller_name',
       'seller_rating', 'driver_rating', 'driver_reviews_num', 'price_drop',
       'price', 'had_price_drop', 'seller_rating_missing', 'mpg_city',
       'mpg_hwy', 'transmission_type', 'age', 'miles_per_year'],
      dtype='str')

In [3]:
features = ["manufacturer", "model", "age", "mileage", "miles_per_year",
    "drivetrain", "fuel_type", "transmission_type",
    "accidents_or_damage", "one_owner", "personal_use_only",
    "mpg_city", "mpg_hwy", "driver_rating", "driver_reviews_num",
    "seller_rating", "seller_rating_missing"]

cat_cols = ["manufacturer", "model", "drivetrain", "fuel_type", "transmission_type",
            "accidents_or_damage", "one_owner", "personal_use_only"]

df[cat_cols] = df[cat_cols].astype("category")

In [4]:
train, test = train_test_split(df, test_size=0.2, random_state=42)
train_fit, valid = train_test_split(train, test_size=0.1, random_state=42)
print(train_fit.shape, valid.shape, test.shape)

(539147, 22) (59906, 22) (149764, 22)


In [5]:
model = lgb.LGBMRegressor(
    n_estimators=5000,
    learning_rate=0.05,
    num_leaves=127,
    min_child_samples=20,
    subsample=0.8, subsample_freq=1,
    colsample_bytree=0.8,
    random_state=42,
)

model.fit(
    train_fit[features], np.log(train_fit["price"]),
    eval_set=[(valid[features], np.log(valid["price"]))],
    callbacks=[lgb.early_stopping(100), lgb.log_evaluation(200)],
)

[LightGBM] [Warning] Categorical features with more bins than the configured maximum bin number found.
[LightGBM] [Warning] For categorical features, max_bin and max_bin_by_feature may be ignored with a large number of categories.


c:\Users\Manci\Desktop\GitHub projects\used-car-deal-detector\.venv\Lib\site-packages\lightgbm\sklearn.py:1106: LGBMDeprecationWarning: The argument 'eval_set' is deprecated, use 'eval_X' and 'eval_y' instead.
  eval_set = _validate_eval_set_Xy(eval_set=eval_set, eval_X=eval_X, eval_y=eval_y)


[LightGBM] [Info] Auto-choosing row-wise multi-threading, the overhead of testing was 0.012374 seconds.
You can set `force_row_wise=true` to remove the overhead.
And if memory is not enough, you can set `force_col_wise=true`.
[LightGBM] [Info] Total Bins 5761
[LightGBM] [Info] Number of data points in the train set: 539147, number of used features: 17
[LightGBM] [Info] Start training from score 10.218138
Training until validation scores don't improve for 100 rounds
[200]	valid_0's l2: 0.0147442
[400]	valid_0's l2: 0.0140911
[600]	valid_0's l2: 0.0138743
[800]	valid_0's l2: 0.0137448
[1000]	valid_0's l2: 0.0136605
[1200]	valid_0's l2: 0.0135837
[1400]	valid_0's l2: 0.0135302
[1600]	valid_0's l2: 0.013503
[1800]	valid_0's l2: 0.0134638
[2000]	valid_0's l2: 0.01344
[2200]	valid_0's l2: 0.0134254
[2400]	valid_0's l2: 0.0134194
[2600]	valid_0's l2: 0.0134109
[2800]	valid_0's l2: 0.0134027
Early stopping, best iteration is:
[2793]	valid_0's l2: 0.0134016


,num_leaves,127
,learning_rate,0.05
,n_estimators,5000
,subsample,0.8
,subsample_freq,1
,colsample_bytree,0.8
,random_state,42
,boosting_type,'gbdt'
,max_depth,-1
,subsample_for_bin,200000
,objective,None


In [6]:
pred = np.exp(model.predict(test[features]))

err = (pred - test["price"]).abs()
print(f"MAE:  ${err.mean():,.0f}")
print(f"MAPE: {(err / test['price']).mean():.1%}")
print(f"Median abs error: ${err.median():,.0f}")

MAE:  $2,219
MAPE: 7.7%
Median abs error: $1,421


In [7]:
importance = pd.Series(
    model.booster_.feature_importance("gain"), index=features
).sort_values(ascending=False)
(importance / importance.sum()).round(3)

age                      0.292
mileage                  0.191
model                    0.159
mpg_hwy                  0.118
manufacturer             0.072
drivetrain               0.046
driver_reviews_num       0.046
mpg_city                 0.033
miles_per_year           0.021
fuel_type                0.007
driver_rating            0.006
seller_rating            0.004
accidents_or_damage      0.002
transmission_type        0.001
personal_use_only        0.001
one_owner                0.001
seller_rating_missing    0.000
dtype: float64

In [8]:
results = test.assign(pred=pred, abs_pct_err=err / test["price"])

# By price range
results["price_band"] = pd.cut(results["price"], [0, 10_000, 20_000, 35_000, 60_000, 200_000])
print(results.groupby("price_band", observed=True)["abs_pct_err"].agg(["mean", "count"]).round(3))

# Worst makes (with at least 500 test cars)
by_make = results.groupby("manufacturer", observed=True)["abs_pct_err"].agg(["mean", "count"])
print(by_make[by_make["count"] >= 500].sort_values("mean", ascending=False).head(10).round(3))

                  mean  count
price_band                   
(0, 10000]       0.204   7786
(10000, 20000]   0.095  33397
(20000, 35000]   0.064  60081
(35000, 60000]   0.059  38052
(60000, 200000]  0.073  10448
                mean  count
manufacturer               
Jaguar         0.101    682
Dodge          0.099   5015
Mitsubishi     0.088   1128
Porsche        0.087   2167
Chrysler       0.087   2447
Ford           0.087  15594
Chevrolet      0.085  10970
Cadillac       0.085   3556
Nissan         0.080   9524
Mercedes-Benz  0.080   8057
